# 13 — Structural breaks

Test pre-specified breaks at 2013, 2021 and 2022. With only 20 annual observations, keep the model deliberately small and treat break tests as diagnostics.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from portugal_refining_resilience.config import get_paths, load_analysis_config
from portugal_refining_resilience.io import persist_dataframe, write_json

PATHS = get_paths(ROOT)
pd.set_option("display.max_columns", 100)

from portugal_refining_resilience.breaks import (
    andrews_sup_wald,
    chow_test,
    coefficient_table,
    interrupted_time_series,
    residual_diagnostics,
)


In [ ]:
panel = pd.read_csv(PATHS.processed / "fuel_annual_analytical_panel.csv")
transition_years = (2021,)
_config = load_analysis_config(ROOT)["annual_break_tests"]
PRE_SPECIFIED = set(_config["pre_specified_break_years"])
# The windows below are stated here, so guard against them drifting from config.
BREAK_WINDOWS = [(2000, 1990, 2012), (2013, 2000, 2020), (2022, 2013, 2024)]
if {year for year, _, _ in BREAK_WINDOWS} != set(_config["candidate_break_years"]):
    raise ValueError("Break windows and config candidate_break_years disagree")
records = []
for product, sub in panel.groupby("product"):
    for outcome in ["exports_kt", "imports_kt", "refinery_output_kt", "net_import_to_demand_ratio"]:
        if outcome not in sub or sub[outcome].notna().sum() < 10:
            continue
        # Each candidate break is tested inside a window that reaches back to the
        # previous candidate, so a test is never contaminated by an earlier event.
        for break_year, window_start, window_end in BREAK_WINDOWS:
            event_sub = sub.loc[sub["year"].between(window_start, window_end)]
            try:
                result = chow_test(event_sub["year"], event_sub[outcome], break_year=break_year, transition_years=transition_years)
            except ValueError:
                continue
            records.append({
                "product": product, "outcome": outcome, "break_year": break_year,
                "f_statistic": result.f_statistic, "p_value": result.p_value,
                "n_pre": result.n_pre, "n_post": result.n_post,
                "window_start": window_start, "window_end": window_end,
                "excluded_years": ",".join(str(year) for year in result.excluded_years),
                "method": "Chow linear-trend diagnostic",
                # 2000 was added after the panel was extended; the report says so and
                # the artifact must too, or a downstream reader inherits the wrong claim.
                "specification": (
                    "pre-specified" if break_year in PRE_SPECIFIED else "exploratory"
                ),
            })
breaks = pd.DataFrame(records)
if not breaks.empty:
    breaks = breaks.sort_values("p_value").reset_index(drop=True)
    m = len(breaks)
    raw_bh = breaks["p_value"] * m / (breaks.index + 1)
    breaks["bh_fdr_p_value"] = raw_bh.iloc[::-1].cummin().iloc[::-1].clip(upper=1.0)
    # Benjamini-Hochberg assumes the tests are independent or positively dependent.
    # They are neither: net imports over demand is a deterministic function of the
    # imports and exports tested beside it, so the same series is being read three
    # ways. Benjamini-Yekutieli holds under arbitrary dependence at the cost of a
    # sum(1/i) penalty, and reporting both shows which results need the assumption.
    by_penalty = float(np.sum(1.0 / np.arange(1, m + 1)))
    raw_by = breaks["p_value"] * m * by_penalty / (breaks.index + 1)
    breaks["by_fdr_p_value"] = raw_by.iloc[::-1].cummin().iloc[::-1].clip(upper=1.0)
    breaks["survives_bh_5pct"] = breaks["bh_fdr_p_value"] < 0.05
    breaks["survives_by_5pct"] = breaks["by_fdr_p_value"] < 0.05
    breaks["by_penalty_factor"] = by_penalty
persist_dataframe(breaks, PATHS.metrics / "structural_break_tests.csv")
display(breaks.sort_values("p_value").head(20))


In [ ]:
# Event-aligned interrupted trend models: coefficients are saved, not copied by hand.
coef_rows = []
its_diagnostics = []
for product, sub in panel.groupby("product"):
    for outcome in ["exports_kt", "net_import_to_demand_ratio"]:
        if outcome not in sub or sub[outcome].notna().sum() < 10:
            continue
        for event_year in [2013, 2022]:
            # A 2022 model fitted on the whole panel draws its counterfactual trend
            # through the 2013 hydrocracker, which is the largest feature of the diesel
            # export series. Both specifications are kept: the reader needs to see that
            # the 2022 export shift does not survive controlling for 2013.
            for specification, controls in (
                ("single event", ()),
                ("controlling for 2013", (2013,) if event_year != 2013 else ()),
            ):
                if specification == "controlling for 2013" and not controls:
                    continue
                model = interrupted_time_series(
                    sub,
                    value_column=outcome,
                    event_year=event_year,
                    transition_years=transition_years,
                    control_events=controls,
                )
                table = coefficient_table(
                    model,
                    product=product,
                    outcome=outcome,
                    event_year=event_year,
                    specification=specification,
                    covariance="HAC(1)",
                    interpretation="associational interrupted trend",
                )
                coef_rows.extend(table.to_dict("records"))
                its_diagnostics.append(
                    residual_diagnostics(
                        model,
                        label=f"{product}/{outcome}/{event_year}/{specification}",
                    )
                )
coef = pd.DataFrame(coef_rows)
persist_dataframe(coef, PATHS.metrics / "annual_interrupted_trend_models.csv")
its_diagnostic_frame = pd.DataFrame(its_diagnostics)
persist_dataframe(its_diagnostic_frame, PATHS.metrics / "annual_model_residual_diagnostics.csv", key_columns=["model"])
display(its_diagnostic_frame)

display(coef.head(20))


In [ ]:
# A Chow test is valid at a break specified in advance. 2000 was added after the panel
# was extended and the series was seen, so its F has to be judged against the
# distribution of the maximum over candidate years, not against a single test.
sup_wald_rows = []
_exploratory_window = (1990, 2012)
for product, sub in panel.groupby("product"):
    window = sub.loc[sub["year"].between(*_exploratory_window)]
    for outcome in ["imports_kt", "refinery_output_kt", "exports_kt", "net_import_to_demand_ratio"]:
        if outcome not in window or window[outcome].notna().sum() < 12:
            continue
        result = andrews_sup_wald(window, value_column=outcome)
        sup_wald_rows.append({
            "product": product,
            "outcome": outcome,
            "window_start": _exploratory_window[0],
            "window_end": _exploratory_window[1],
            "sup_wald_statistic": result.statistic,
            "break_year_selected": result.break_year,
            "p_value": result.p_value,
            "null_95th_percentile": result.null_95th_percentile,
            "n_candidates": result.n_candidates,
            "n_simulations": result.n_simulations,
            "significant_5pct": bool(result.p_value < 0.05),
            "method": "Andrews sup-Wald, simulated null",
        })
sup_wald = pd.DataFrame(sup_wald_rows)
persist_dataframe(sup_wald, PATHS.metrics / "exploratory_break_sup_wald.csv", key_columns=["product", "outcome"])
display(sup_wald)
